# Train CNN
*This notebook aims to train a CNN model to classify images (pokemon 2D pictures)*

## Summary

- [Imports & Configuration](#imports--configuration)
- [Data Preparation](#data-preparation)
- [Model](#model)

## Imports & Configuration

In [1]:
import os
import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.set_experiment("final-project")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

2025/07/10 18:16:54 INFO mlflow.tracking.fluent: Experiment with name 'final-project' does not exist. Creating a new experiment.


Using device: cpu


## Data Preparation

In [11]:
from pathlib import Path
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

# Recherche du dossier data/raw dans les parents
cwd = Path.cwd()
raw_dir = None
for ancestor in [cwd] + list(cwd.parents):
    candidate = ancestor / "data" / "raw"
    if candidate.is_dir():
        raw_dir = candidate
        break

if raw_dir is None:
    raise FileNotFoundError(f"Impossible de trouver le dossier 'data/raw' depuis {cwd}")

print(f"→ RAW_DIR détecté : {raw_dir}")

# Hyper-paramètres
BATCH_SIZE  = 32
IMG_SIZE    = 224
TRAIN_RATIO = 0.8
SEED        = 42

# Transforms
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])
val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

# Chargement complet
full_dataset = ImageFolder(str(raw_dir))
class_names  = full_dataset.classes
n_total      = len(full_dataset)
n_train      = int(n_total * TRAIN_RATIO)
n_val        = n_total - n_train

# Split aléatoire
torch.manual_seed(SEED)
train_subset, val_subset = random_split(full_dataset, [n_train, n_val])

# Assigner les transforms respectifs
train_subset.dataset.transform = train_transforms
val_subset.dataset.transform   = val_transforms

# Création des DataLoaders
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Total images: {n_total} → Train: {n_train}, Val: {n_val}")
print(f"Classes détectées: {class_names}")


→ RAW_DIR détecté : c:\Users\marco\Desktop\devops-pokefind\data\raw
Total images: 10657 → Train: 8525, Val: 2132
Classes détectées: ['Abra', 'Aerodactyl', 'Alakazam', 'Arbok', 'Arcanine', 'Articuno', 'Beedrill', 'Bellsprout', 'Blastoise', 'Bulbasaur', 'Butterfree', 'Caterpie', 'Chansey', 'Charizard', 'Charmander', 'Charmeleon', 'Clefable', 'Clefairy', 'Cloyster', 'Cubone', 'Dewgong', 'Diglett', 'Ditto', 'Dodrio', 'Doduo', 'Dragonair', 'Dragonite', 'Dratini', 'Drowzee', 'Dugtrio', 'Eevee', 'Ekans', 'Electabuzz', 'Electrode', 'Exeggcute', 'Exeggutor', 'Farfetchd', 'Fearow', 'Flareon', 'Gastly', 'Gengar', 'Geodude', 'Gloom', 'Golbat', 'Goldeen', 'Golduck', 'Golem', 'Graveler', 'Grimer', 'Growlithe', 'Gyarados', 'Haunter', 'Hitmonchan', 'Hitmonlee', 'Horsea', 'Hypno', 'Ivysaur', 'Jigglypuff', 'Jolteon', 'Jynx', 'Kabuto', 'Kabutops', 'Kadabra', 'Kakuna', 'Kangaskhan', 'Kingler', 'Koffing', 'Krabby', 'Lapras', 'Lickitung', 'Machamp', 'Machoke', 'Machop', 'Magikarp', 'Magmar', 'Magnemite', 'M

## Model

In [ ]:
# On part d'un modèle pré-entraîné
model = models.resnet18(pretrained=True)

# Adaptation de la dernière couche au nombre de classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(train_dataset.classes))

model = model.to(device)
